# 02. Exploratory Data Analysis

This notebook explores the generated development fixture only. It compares available supply with open demand, checks monthly patterns and the matching funnel, and summarizes village fulfillment. All counts and charts are synthetic; use your separately consented pilot export for real findings.

In [ ]:
from pathlib import Path
import json
import pandas as pd

cwd = Path.cwd()
WORKSPACE = next((path for path in (cwd, *cwd.parents) if (path / "data" / "synthetic").exists()), cwd)
DATA_DIR = WORKSPACE / "data" / "synthetic"
items = pd.read_csv(DATA_DIR / "items.csv")
needs = pd.read_csv(DATA_DIR / "needs.csv")
matches = pd.read_csv(DATA_DIR / "matches.csv")
villages = pd.read_csv(DATA_DIR / "villages.csv")
metadata = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
assert all(set(frame["data_origin"].dropna()) == {"synthetic"} for frame in [items, needs, matches, villages])
items["created_at"] = pd.to_datetime(items["created_at"], utc=True)
needs["created_at"] = pd.to_datetime(needs["created_at"], utc=True)
print(f"Loaded synthetic fixture, seed={metadata['seed']}, as_of={metadata['as_of']}.")
print(f"Items: {len(items):,}; needs: {len(needs):,}; candidate matches: {len(matches):,}.")

In [ ]:
# Supply versus currently open demand by category
import matplotlib.pyplot as plt

supply = items.loc[items["item_status"] == "approved"].groupby("category")["quantity"].sum().rename("approved_supply")
demand = needs.loc[needs["need_status"] == "open"].groupby("category")["quantity"].sum().rename("open_demand")
category_gap = pd.concat([supply, demand], axis=1).fillna(0).astype(int).sort_index()
category_gap.plot(kind="bar", figsize=(11, 4), color=["#2f7f6f", "#d28a3b"])
plt.title("Synthetic approved supply and open demand by category")
plt.ylabel("Units (synthetic)")
plt.xlabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()
display(category_gap.assign(net_gap=category_gap["open_demand"] - category_gap["approved_supply"]))

In [ ]:
# Monthly trend and seasonality check
import matplotlib.pyplot as plt

items["month"] = items["created_at"].dt.tz_localize(None).dt.to_period("M").astype(str)
needs["month"] = needs["created_at"].dt.tz_localize(None).dt.to_period("M").astype(str)
monthly = pd.concat([
    items.groupby("month").size().rename("item_listings"),
    needs.groupby("month").size().rename("needs_posted"),
], axis=1).fillna(0).sort_index()
monthly.plot(kind="line", marker="o", figsize=(11, 4), color=["#2f7f6f", "#d28a3b"])
plt.title("Synthetic item listings and needs by month")
plt.ylabel("Records (synthetic)")
plt.xlabel("Month")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()
display(monthly)

In [ ]:
# Item-level funnel and village fulfillment
import matplotlib.pyplot as plt

accepted_states = {"accepted", "pickup_scheduled", "in_transit", "delivered", "confirmed"}
delivered_states = {"delivered", "confirmed"}
funnel = pd.Series({
    "uploaded items": items["id"].nunique(),
    "items with a suggestion": matches["item_id"].nunique(),
    "items with an accepted match": matches.loc[matches["match_status"].isin(accepted_states), "item_id"].nunique(),
    "items delivered": matches.loc[matches["match_status"].isin(delivered_states), "item_id"].nunique(),
    "items confirmed": matches.loc[matches["match_status"] == "confirmed", "item_id"].nunique(),
})
funnel.plot(kind="bar", color="#2f7f6f", figsize=(9, 4))
plt.title("Synthetic item-level matching funnel")
plt.ylabel("Unique items")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

village_outcomes = needs.groupby("village_id").agg(
    needs_posted=("id", "nunique"),
    needs_fulfilled=("need_status", lambda values: int((values == "fulfilled").sum())),
)
village_outcomes["fulfillment_rate"] = village_outcomes["needs_fulfilled"] / village_outcomes["needs_posted"]
village_outcomes = village_outcomes.merge(villages[["id", "name"]], left_index=True, right_on="id").set_index("name")
village_outcomes["unmet_needs"] = village_outcomes["needs_posted"] - village_outcomes["needs_fulfilled"]
display(village_outcomes[["needs_posted", "needs_fulfilled", "unmet_needs", "fulfillment_rate"]].sort_values("fulfillment_rate"))

In [ ]:
# Village coordinate and distance views (only when coordinates were supplied)
import matplotlib.pyplot as plt

coordinate_villages = villages.dropna(subset=["latitude", "longitude"])
if coordinate_villages.empty:
    print("Map and distance analysis skipped: no campus coordinate was supplied to the synthetic generator.")
else:
    plt.figure(figsize=(7, 6))
    plt.scatter(coordinate_villages["longitude"], coordinate_villages["latitude"], color="#d28a3b")
    for row in coordinate_villages.itertuples():
        plt.annotate(row.name.replace("Synthetic Village ", "V"), (row.longitude, row.latitude), xytext=(4, 4), textcoords="offset points")
    plt.title("Synthetic village locations (generated offsets, not surveyed locations)")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.show()

feature_rows = matches["feature_snapshot"].map(json.loads).apply(pd.Series)
if "distance_km" in feature_rows and feature_rows["distance_km"].notna().any():
    distance_view = matches[["match_status"]].join(feature_rows[["distance_km"]])
    distance_view["confirmed"] = (distance_view["match_status"] == "confirmed").astype(int)
    distance_view.plot.scatter(x="distance_km", y="confirmed", alpha=0.25, figsize=(7, 4), color="#2f7f6f")
    plt.title("Synthetic candidate distance versus confirmation")
    plt.ylabel("Confirmed (0/1)")
    plt.tight_layout()
    plt.show()
else:
    print("Distance-versus-fulfillment analysis skipped because the fixture has no coordinates.")

In [ ]:
# Plain-language conclusion, computed from the fixture
net_gap = category_gap["open_demand"] - category_gap["approved_supply"]
most_unmet_category = str(net_gap.idxmax())
fulfillment_rate = float((needs["need_status"] == "fulfilled").mean()) if len(needs) else 0.0
suggested_item_count = int(matches["item_id"].nunique())
print(f"Conclusion (synthetic only): {most_unmet_category} has the largest positive open-demand-minus-approved-supply gap ({int(net_gap.max())} units) in this generated snapshot.")
print(f"The generated need fulfillment rate is {fulfillment_rate:.1%}; {suggested_item_count} unique items received at least one match suggestion.")
print("Limitations: generated seasonality, village profiles, candidate scores, and acceptance outcomes are simulation assumptions. Approved listings are not necessarily available units, needs may overlap, and none of these numbers are pilot results.")